In [ ]:
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.models as models
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

In [ ]:
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Modèle du GPU : {torch.cuda.get_device_name(0)}")

In [ ]:
## This section is for data manipulation ##

def _to_dataframe(path):
    df = pd.read_csv(path)
    label_map = {
        "SNE": 0,
        "LY": 1,
        "MO": 2,
        "EO": 3,
        "BA": 4,
        "VLY": 5,
        "MMY": 6,
        "MY": 7,
        "PMY": 8,
        "BL": 9,
        "PC": 10,
        "PLY": 11,
        "BNE": 0,
    }
    if "label" in df.columns:
        df["label_idx"] = df["label"].map(label_map)
    else:
        df["label"] = "undefined"
        df["label_idx"] = -1
    return df


class WhiteBloodCellDataset(Dataset):
    def __init__(self, dataframe, path, transform=None):
        self.names = dataframe["ID"].values
        self.labels = dataframe["label_idx"].values
        self.path = path
        self.transform = transform

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        image_path = self.path + str(self.names[idx])
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label


def load_data(train_df_path, test_df_path, train_data_path, test_data_path):
    # Resize to 224x224 for ResNet + Data Augmentation
    train_transforms = transforms.Compose(
        [
            transforms.Resize(size=(384, 384)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=180),
            # transforms.ElasticTransform(alpha=50.0, sigma=5.0),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    val_test_transforms = transforms.Compose(
        [
            transforms.Resize((384, 384)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    train_transforms_specialist = transforms.Compose(
        [
            transforms.RandomResizedCrop(size=(384, 384), scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=180),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )

    df_full_train = _to_dataframe(train_df_path)
    df_test = _to_dataframe(test_df_path)
    df_sne_bne = df_full_train[df_full_train["label"].isin(["SNE", "BNE"])].copy()
    df_sne_bne["label_idx"] = df_sne_bne["label"].map({"SNE": 0, "BNE": 1})

    df_train, df_val = train_test_split(
        df_full_train,
        test_size=0.20,
        random_state=0,
        stratify=df_full_train["label_idx"],
    )
    df_train_sne_bne, df_val_sne_bne = train_test_split(
        df_sne_bne,
        test_size=0.20,
        random_state=0,
        stratify=df_sne_bne["label_idx"],
    )

    # Datasets
    train_set = WhiteBloodCellDataset(
        dataframe=df_train,
        path=train_data_path,
        transform=train_transforms,
    )
    val_set = WhiteBloodCellDataset(
        dataframe=df_val,
        path=train_data_path,
        transform=val_test_transforms,
    )
    test_set = WhiteBloodCellDataset(
        dataframe=df_test,
        path=test_data_path,
        transform=val_test_transforms,
    )
    train_sne_bne_set = WhiteBloodCellDataset(
        dataframe=df_train_sne_bne,
        path=train_data_path,
        transform=train_transforms_specialist,
    )
    val_sne_bne_set = WhiteBloodCellDataset(
        dataframe=df_val_sne_bne,
        path=train_data_path,
        transform=val_test_transforms,
    )

    # Sampler
    counts = df_train["label_idx"].value_counts().sort_index().values
    weights = 1.0 / counts
    samples_weights = torch.from_numpy(weights[df_train["label_idx"].values]).double()

    sampler = WeightedRandomSampler(
        weights=samples_weights, num_samples=len(samples_weights), replacement=True
    )

    counts_specialist = df_train_sne_bne["label_idx"].value_counts().sort_index().values
    weights_specialist = 1.0 / counts_specialist
    samples_weights_specialist = torch.from_numpy(weights_specialist[df_train_sne_bne["label_idx"].values]).double()
    
    sampler_specialist = WeightedRandomSampler(
        weights=samples_weights_specialist, num_samples=len(samples_weights_specialist), replacement=True
    )
    # DataLoaders
    train_loader = DataLoader(train_set, batch_size=16, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_set, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    train_sne_bne_loader = DataLoader(train_sne_bne_set, batch_size=32, sampler=sampler_specialist, num_workers=2, pin_memory=True)
    val_sne_bne_loader = DataLoader(val_sne_bne_set, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, test_loader, train_sne_bne_loader, val_sne_bne_loader

In [ ]:
## This section is for model management ##

def get_model():
    model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(num_ftrs, 12)

    return model

    return model

def get_model_specialist():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    for param in model.parameters():
        param.requires_grad = False

    num_ftrs = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.2, inplace=True),
        nn.Linear(num_ftrs, 2)
    )
    
    return model

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.5, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt)**self.gamma * ce_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        else:
            return focal_loss.sum()

def train_model(
    model, device, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, patience, is_specialist
):
    best_f1 = 0
    counter = 0
    history = {
        "train_loss": [],
        "val_f1": [],
    }

    scaler = torch.amp.GradScaler('cuda')
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            
            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            # Stats
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_loss = running_loss / len(train_loader)
        epoch_acc = 100 * correct / total
        history["train_loss"].append(epoch_loss)
        print(
            f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%"
        )

        # Early Stopping
        current_f1 = validate_model(model, device, val_loader)
        scheduler.step(current_f1)
        history["val_f1"].append(current_f1)
        print(f"Epoch [{epoch+1}/{num_epochs}] - Validation Macro-F1: {current_f1:.4f}")

        if current_f1 > best_f1:
            best_f1 = current_f1
            if is_specialist:
                torch.save(model.state_dict(), "best_specialist_white_cell_model.pth")
            else:
                torch.save(model.state_dict(), "best_white_cell_model.pth")
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print(f"Early Stopping !")
            break

    print("Training finished !")
    return history


def validate_model(model, device, test_loader):
    model.eval()
    all_pred = []
    all_labels = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_pred.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    score = f1_score(all_labels, all_pred, average="macro")
    return score


def predict_cascade(model_gen, model_spec, device, test_loader, inv_label_map_gen):
    model_gen.eval()
    model_spec.eval()
    all_preds_strings = []

    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            
            out1 = model_gen(images)
            out2 = model_gen(torch.flip(images, dims=[3]))
            out3 = model_gen(torch.flip(images, dims=[2]))
            out4 = model_gen(torch.rot90(images, k=2, dims=[2, 3]))
            
            outputs_avg_gen = (out1 + out2 + out3 + out4) / 4.0
            _, predicted_gen = torch.max(outputs_avg_gen, 1)

            for i in range(images.size(0)): 
                pred_gen_idx = predicted_gen[i].item()

                if pred_gen_idx == 0:
                    img_single = images[i].unsqueeze(0) 
                    o1 = model_spec(img_single)
                    o2 = model_spec(torch.flip(img_single, dims=[3]))
                    o3 = model_spec(torch.flip(img_single, dims=[2]))
                    o4 = model_spec(torch.rot90(img_single, k=2, dims=[2, 3]))
                    
                    outputs_avg_spec = (o1 + o2 + o3 + o4) / 4.0
                    _, predicted_spec = torch.max(outputs_avg_spec, 1)
                    
                    if predicted_spec.item() == 0:
                        all_preds_strings.append("SNE")
                    else:
                        all_preds_strings.append("BNE")
                
                else:
                    all_preds_strings.append(inv_label_map_gen[pred_gen_idx])

    return all_preds_strings

In [ ]:
import matplotlib.pyplot as plt

def plot_training_curve(train_losses, val_f1s):
    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Train Loss', color='tab:red')
    ax1.plot(train_losses, color='tab:red', label='Loss')
    ax1.tick_params(axis='y', labelcolor='tab:red')

    ax2 = ax1.twinx()
    ax2.set_ylabel('Val Macro-F1', color='tab:blue')
    ax2.plot(val_f1s, color='tab:blue', label='Macro-F1')
    ax2.tick_params(axis='y', labelcolor='tab:blue')

    plt.title('Performance during training')
    plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(model, device, val_loader, labels_names):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels_names, yticklabels=labels_names)
    plt.xlabel('Predictions')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix (Validation Set)')
    plt.show()

In [ ]:
## This section is for training/validation/testing ##

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using : {device}")

# Loading data
train_csv = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train_metadata.csv"
test_csv = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test_metadata.csv"
train_dir = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train/"
test_dir = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test/"
train_loader, val_loader, test_loader, train_sne_bne_loader, val_sne_bne_loader = load_data(
    train_csv, test_csv, train_dir, test_dir
)
print("Data loaded !")

# Specialist Model
model_specialist = get_model_specialist()
model_specialist = model_specialist.to(device)

# Parameters
criterion_specialist = FocalLoss(alpha=0.25, gamma=2.0)
optimizer_specialist_phase1 = optim.Adam(model_specialist.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler_specialist_phase1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_specialist_phase1, mode='max', factor=0.1, patience=2)
num_epochs = 40
patience = 5

history_specialist_phase1 = train_model(
    model_specialist, device, train_sne_bne_loader, val_sne_bne_loader, criterion_specialist, optimizer_specialist_phase1, scheduler_specialist_phase1, num_epochs, patience, True
)
print("First phase of training completed !")

model_specialist.load_state_dict(torch.load("best_specialist_white_cell_model.pth"))

for param in model_specialist.features[7:].parameters():
    param.requires_grad = True

optimizer_specialist_phase2 = optim.Adam([
    {'params': model_specialist.features[7:].parameters(), 'lr': 1e-5, 'weight_decay': 1e-4}, 
    {'params': model_specialist.classifier.parameters(), 'lr': 1e-5, 'weight_decay': 1e-4}
])

scheduler_specialist_phase2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_specialist_phase2, mode='max', factor=0.1, patience=2
)
history_specialist_phase2 = train_model(
    model_specialist, device, train_sne_bne_loader, val_sne_bne_loader, criterion_specialist, optimizer_specialist_phase2, scheduler_specialist_phase2, num_epochs, patience, True
)
model_specialist.load_state_dict(torch.load("best_specialist_white_cell_model.pth"))
print("Second phase of training completed !")
print("Training done !")

# Validation
score = validate_model(model_specialist, device, val_sne_bne_loader)
print(f"The Macro F1-score for Validation is {score}.")

In [ ]:
plot_training_curve(history_specialist_phase1["train_loss"]+history_specialist_phase2["train_loss"], history_specialist_phase1["val_f1"]+history_specialist_phase2["val_f1"])

In [ ]:
plot_confusion_matrix(model_specialist, device, val_sne_bne_loader, ["SNE", "BNE"])

In [ ]:
# Model Generalist
model = get_model()
model = model.to(device)

# Parameters
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_phase1 = optim.Adam(model.parameters(), lr=0.0001)
scheduler_phase1 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_phase1, mode='max', factor=0.1, patience=2)
num_epochs = 40
patience = 5

# Train
history_phase1 = train_model(
    model, device, train_loader, val_loader, criterion, optimizer_phase1, scheduler_phase1, num_epochs, patience, False
)
print("First phase of training completed !")

model.load_state_dict(torch.load("best_white_cell_model.pth"))

for param in model.features[6:].parameters():
    param.requires_grad = True

num_epochs = 30
optimizer_phase2 = optim.Adam([
    {'params': model.features[6:].parameters(), 'lr': 1e-5},
    {'params': model.classifier[1].parameters(), 'lr': 1e-5}
])

scheduler_phase2 = optim.lr_scheduler.ReduceLROnPlateau(optimizer_phase2, mode='max', factor=0.1, patience=2)
history_phase2 = train_model(
    model, device, train_loader, val_loader, criterion, optimizer_phase2, scheduler_phase2, num_epochs, patience, False
)
model.load_state_dict(torch.load("best_white_cell_model.pth"))
print("Second phase of training completed !")
print("Training done !")

# Validation
score = validate_model(model, device, val_loader)
print(f"The Macro F1-score for Validation is {score}.")



# Predictions
inv_label_map_final = {
    0: "SNE", 1: "LY", 2: "MO", 3: "EO", 4: "BA", 5: "VLY", 
    6: "MMY", 7: "MY", 8: "PMY", 9: "BL", 10: "PC", 11: "PLY", 
    12: "BNE" 
}
predictions = predict_cascade(model, model_specialist, device, test_loader, inv_label_map)
sample_sub = pd.read_csv("/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/sample_submission.csv")
if len(sample_sub) == len(predictions):
    sample_sub = sample_sub.drop(columns=["Unnamed: 0"])
    sample_sub["label"] = predictions
    sample_sub.to_csv("submission.csv", index=False)
    print("Fichier submission.csv prêt !")
else:
    print(f"Size error ! Got: {len(predictions)} instead of: {len(sample_sub)}")
print("Model tested !")

In [ ]:
plot_training_curve(history_phase1["train_loss"]+history_phase2["train_loss"], history_phase1["val_f1"]+history_phase2["val_f1"])

In [ ]:
plot_confusion_matrix(model, device, val_loader, ["SNE/BNE","LY","MO","EO","BA","VLY","MMY","MY","PMY","BL","PC","PLY"])